# VITMAV45 – Komplex architektúrák építőelemei: gyakorlat (hallgatói verzió)

**90 perc · Google Colab · Keras 3 · egyéni munka.**

Három feladat, mindegyik két részből áll:
- **A rész – kidolgozott példa:** egy egyszerűbb feladaton megnézzük a megoldást. Futtasd a cellákat sorban, és olvasd el a magyarázatokat.
- **B rész – saját feladat:** egy hasonló feladatot neked kell megoldanod az A rész mintájára. A `# TODO` helyekre kell kódot írnod, utána egy `check_...()` cella jelzi, hogy jó-e.

| Feladat | A rész (kidolgozott) | B rész (saját) | Előadásdiák |
|---|---|---|---|
| 1 | Residual kapcsolat mély hálóban | Highway-réteg (kapuzott skip) | 12–21, 26 |
| 2 | Mozgó átlag maszkolt softmaxszal | Maszkolt self-attention | 29–32 |
| 3 | Tanulás a teacher puha címkéiből | Desztilláció hőmérséklettel | 46–47 |

## 0. Indítás

Futtasd le a következő két cellát. Az első betölti a könyvtárakat, a második az ellenőrző függvényeket (ezt nem kell elolvasnod).

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"     # a keras importja ELŐTT kell beállítani

import math
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers, ops

assert keras.__version__.startswith("3"), "Keras 3 kell (Colabban alapértelmezett)"
keras.utils.set_random_seed(0)
print("Keras:", keras.__version__, "| háttér:", keras.backend.backend())

RESULTS = {}   # ide gyűjtjük a mért számokat

def ok(msg):
    print("✅", msg)

In [ ]:
# Ellenőrző függvények (nem kell módosítani, csak lefuttatni)
import base64
exec(base64.b64decode("IyBbRUxMRU7FkFJaxZAgQ0VMTEEg4oCTIGEgaGFsbGdhdMOzaSB2ZXJ6acOzYmFuIGvDs2RvbHZhIGplbGVuaWsgbWVnXQojIEVsbGVuxZFyesWRIGbDvGdndsOpbnllayAobmVtIGtlbGwgbcOzZG9zw610YW5pLCBjc2FrIGxlZnV0dGF0bmkpCgpkZWYgX25wKHgpOgogICAgcmV0dXJuIG5wLmFzYXJyYXkob3BzLmNvbnZlcnRfdG9fbnVtcHkoeCkpCgpkZWYgX3NpZ21vaWQoeik6CiAgICByZXR1cm4gMSAvICgxICsgbnAuZXhwKC16KSkKCmRlZiBjaGVja19oaWdod2F5KCk6CiAgICBrZXJhcy51dGlscy5zZXRfcmFuZG9tX3NlZWQoMSkKICAgIHggPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoMSkubm9ybWFsKHNpemU9KDUsIDIpKS5hc3R5cGUoImZsb2F0MzIiKQogICAgbSA9IGJ1aWxkX2hpZ2h3YXkoZGVwdGg9Mywgd2lkdGg9NCkKICAgIFcgPSB7bC5uYW1lOiBsLmdldF93ZWlnaHRzKCkgZm9yIGwgaW4gbS5sYXllcnMgaWYgbC5nZXRfd2VpZ2h0cygpfQogICAgZm9yIGkgaW4gcmFuZ2UoMyk6CiAgICAgICAgYXNzZXJ0IGYiSF97aX0iIGluIFcgYW5kIGYiVF97aX0iIGluIFcsICJBIHLDqXRlZ2VrIG5ldmUgbGVneWVuIEhfMCwgVF8wLCBIXzEsIFRfMSwg4oCmIgogICAgICAgIGFzc2VydCBucC5hbGxjbG9zZShXW2YiVF97aX0iXVsxXSwgLTMuMCksICJBIGthcHUgKFQpIGJpYXNhIGluZHVsam9uIOKIkjMtcsOzbCIKICAgIGggPSBucC50YW5oKHggQCBXWyJpbnAiXVswXSArIFdbImlucCJdWzFdKQogICAgZm9yIGkgaW4gcmFuZ2UoMyk6CiAgICAgICAgSCA9IG5wLnRhbmgoaCBAIFdbZiJIX3tpfSJdWzBdICsgV1tmIkhfe2l9Il1bMV0pCiAgICAgICAgVCA9IF9zaWdtb2lkKGggQCBXW2YiVF97aX0iXVswXSArIFdbZiJUX3tpfSJdWzFdKQogICAgICAgIGggPSBIICogVCArIGggKiAoMSAtIFQpCiAgICByZWYgPSBfc2lnbW9pZChoIEAgV1sib3V0Il1bMF0gKyBXWyJvdXQiXVsxXSkKICAgIGFzc2VydCBucC5hbGxjbG9zZShfbnAobSh4KSksIHJlZiwgYXRvbD0xZS01KSwgInkgPSBIwrdUICsgaMK3KDEg4oiSIFQpIGtlbGwsIEg6IHRhbmgsIFQ6IHNpZ21vaWQiCiAgICBvaygiYnVpbGRfaGlnaHdheSByZW5kYmVuIikKCgpkZWYgX25wX2F0dGVudGlvbihxLCBrLCB2KToKICAgIG4gPSBxLnNoYXBlWy0yXQogICAgcyA9IHEgQCBucC5zd2FwYXhlcyhrLCAtMSwgLTIpIC8gbWF0aC5zcXJ0KHEuc2hhcGVbLTFdKQogICAgcyA9IG5wLndoZXJlKG5wLnRyaXUobnAub25lcygobiwgbiksIGJvb2wpLCAxKSwgLW5wLmluZiwgcykKICAgIHcgPSBucC5leHAocyAtIHMubWF4KC0xLCBrZWVwZGltcz1UcnVlKSkKICAgIHcgLz0gdy5zdW0oLTEsIGtlZXBkaW1zPVRydWUpCiAgICByZXR1cm4gdyBAIHYsIHcKCgpkZWYgY2hlY2tfYXR0ZW50aW9uKCk6CiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoMCkKICAgIHEsIGssIHYgPSAocm5nLm5vcm1hbChzaXplPSgyLCA3LCA4KSkuYXN0eXBlKCJmbG9hdDMyIikgZm9yIF8gaW4gcmFuZ2UoMykpCiAgICBvdXQsIHcgPSBtYXNrZWRfYXR0ZW50aW9uKHEsIGssIHYpCiAgICByZWZfb3V0LCByZWZfdyA9IF9ucF9hdHRlbnRpb24ocSwgaywgdikKICAgIGFzc2VydCBfbnAodykuc2hhcGUgPT0gKDIsIDcsIDcpLCBmIkEgc8O6bHltw6F0cml4IGFsYWtqYSAo4oCmLCBuLCBuKSBsZWd5ZW4sIG1vc3Qge19ucCh3KS5zaGFwZX0iCiAgICBhc3NlcnQgbnAuYWxsY2xvc2UoX25wKHcpLCByZWZfdywgYXRvbD0xZS01KSwgIkEgc8O6bHlvayBlbHTDqXJuZWs6IFFL4bWAL+KImmQsIG1hc3prIGEgc29mdG1heCBFTMWQVFQiCiAgICBhc3NlcnQgbnAuYWxsY2xvc2UoX25wKG91dCksIHJlZl9vdXQsIGF0b2w9MWUtNSksICJBIGtpbWVuZXQgZWx0w6lyOiBzw7pseW9rIEAgViIKICAgIG9rKCJtYXNrZWRfYXR0ZW50aW9uIHJlbmRiZW4iKQoKCmRlZiBjaGVja19jYXVzYWwoZiwgeCwgdDApOgogICAgIyB4OiAoQiwgVCwgQykuIElnYXosIGhhIGEgdDAgdXTDoW5pIGJlbWVuZXQgbWVndsOhbHRvenRhdMOhc2EgYSB0MC1pZyB0YXJ0w7Mga2ltZW5ldGV0IE5FTSB2w6FsdG96dGF0amEuCiAgICB5MSA9IF9ucChmKHgpKQogICAgeDIgPSB4LmNvcHkoKQogICAgeDJbOiwgdDAgKyAxOl0gPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoNSkubm9ybWFsKHNpemU9eDJbOiwgdDAgKyAxOl0uc2hhcGUpCiAgICB5MiA9IF9ucChmKHgyKSkKICAgIHNhbWVfcGFzdCA9IG5wLmFsbGNsb3NlKHkxWzosIDp0MCArIDFdLCB5Mls6LCA6dDAgKyAxXSwgYXRvbD0xZS01KQogICAgY2hhbmdlZF9mdXR1cmUgPSBub3QgbnAuYWxsY2xvc2UoeTFbOiwgdDAgKyAxOl0sIHkyWzosIHQwICsgMTpdLCBhdG9sPTFlLTUpCiAgICByZXR1cm4gc2FtZV9wYXN0IGFuZCBjaGFuZ2VkX2Z1dHVyZQoKCmRlZiBjaGVja19sb2NhbF9tYXNrKCk6CiAgICBtID0gX25wKGxvY2FsX21hc2soNSwgMikpLmFzdHlwZShib29sKQogICAgZXhwID0gbnAuYXJyYXkoW1swLCAxLCAxLCAxLCAxXSwgWzAsIDAsIDEsIDEsIDFdLCBbMSwgMCwgMCwgMSwgMV0sIFsxLCAxLCAwLCAwLCAxXSwgWzEsIDEsIDEsIDAsIDBdXSwgYm9vbCkKICAgIGFzc2VydCBucC5hcnJheV9lcXVhbChtLCBleHApLCAiVHJ1ZSA9IG1hc3prb2x0OiBhIGrDtnbFkSDDiVMgYSB3LW7DqWwgcsOpZ2ViYmkgbcO6bHQiCiAgICBvaygibG9jYWxfbWFzayByZW5kYmVuIikKCgpkZWYgY2hlY2tfa2RfbG9zcygpOgogICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKDApCiAgICBzLCB0ID0gcm5nLm5vcm1hbChzaXplPSg2LCAxMCkpLmFzdHlwZSgiZmxvYXQzMiIpLCBybmcubm9ybWFsKHNpemU9KDYsIDEwKSkuYXN0eXBlKCJmbG9hdDMyIikKICAgIFQgPSAzLjAKICAgIGRlZiBzbSh6KToKICAgICAgICBlID0gbnAuZXhwKHogLSB6Lm1heCgtMSwga2VlcGRpbXM9VHJ1ZSkpOyByZXR1cm4gZSAvIGUuc3VtKC0xLCBrZWVwZGltcz1UcnVlKQogICAgcF90LCBwX3MgPSBzbSh0IC8gVCksIHNtKHMgLyBUKQogICAgcmVmID0gKHBfdCAqIChucC5sb2cocF90KSAtIG5wLmxvZyhwX3MpKSkuc3VtKC0xKS5tZWFuKCkgKiBUICogVAogICAgYXNzZXJ0IGFicyhmbG9hdChfbnAoa2RfbG9zcyhzLCB0LCBUKSkpIC0gcmVmKSA8IDFlLTMsICJLRC1sb3NzOiBUwrIgwrcgS0woc29mdG1heCh0L1QpIOKAliBzb2Z0bWF4KHMvVCkpLCBiYXRjaC3DoXRsYWciCiAgICBhc3NlcnQgYWJzKGZsb2F0KF9ucChrZF9sb3NzKHQsIHQsIFQpKSkpIDwgMWUtNCwgIkF6b25vcyBsb2dpdG9rcmEgYSBLRC1sb3NzIDAiCiAgICBvaygia2RfbG9zcyByZW5kYmVuIikKCm9rKCJlbGxlbsWRcnrFkSBmw7xnZ3bDqW55ZWsgYmV0w7ZsdHZlIik=").decode("utf-8"))

---
# 1. feladat · Skip kapcsolatok mély hálóban

Az előadás Keras-os XOR-példáját (12–17. dia) folytatjuk. Kérdés: mitől tanul meg egy nagyon mély háló?

## 1A · Kidolgozott példa: residual kapcsolat

**Az adat:** 2D XOR. A címke 1, ha a két koordináta azonos előjelű.

In [ ]:
rng = np.random.default_rng(0)
X_xor = rng.uniform(-1, 1, size=(512, 2)).astype("float32")
y_xor = ((X_xor[:, 0] * X_xor[:, 1]) > 0).astype("float32")[:, None]

plt.figure(figsize=(3, 3))
plt.scatter(X_xor[:, 0], X_xor[:, 1], c=y_xor[:, 0], cmap="coolwarm", s=6)
plt.title("XOR"); plt.show()

**A modell.** Egy bemeneti réteg (`inp`), utána `depth` darab tanh-réteg, a végén egy szigmoid kimenet.

Egy rejtett réteg: `z = tanh(W·h + b)`.
- Sima hálóban a következő réteg `h ← z`.
- **Residual** hálóban a következő réteg `h ← h + z`: a bemenet változatlanul átmegy, a réteg csak „hozzáad” valamit. Kerasban ez a `layers.Add()`.

In [ ]:
def build_deep(depth, width=16, residual=False):
    inp = keras.Input(shape=(2,))
    h = layers.Dense(width, activation="tanh", name="inp")(inp)
    for i in range(depth):
        z = layers.Dense(width, activation="tanh", name=f"hidden_{i}")(h)
        h = layers.Add()([h, z]) if residual else z           # ← ez az egyetlen különbség
    out = layers.Dense(1, activation="sigmoid", name="out")(h)
    return keras.Model(inp, out)

**Mit mérünk?** Az `inp` réteg (a háló legeleje) súlyainak gradiensét, tanítás előtt. Ha ez nagyon kicsi, a háló eleje alig tanul: ez az eltűnő gradiens (vanishing gradient).

In [ ]:
def first_layer_grad_norm(model, X, y, loss_fn):
    kernel = model.get_layer("inp").kernel
    with tf.GradientTape() as tape:
        loss = loss_fn(y, model(X))
    (g,) = tape.gradient(loss, [kernel])
    return float(tf.norm(g))

**Tanítás.** Egy rövid segédfüggvény: lefordítja a modellt, tanít, és visszaadja a végső veszteséget.

In [ ]:
def fit_xor(model, epochs=200):
    model.compile(optimizer=keras.optimizers.Adam(3e-3), loss="mse")
    hist = model.fit(X_xor, y_xor, epochs=epochs, batch_size=len(X_xor), verbose=0)
    return hist.history["loss"]

mse = keras.losses.MeanSquaredError()

**Kísérlet:** 10 és 50 rejtett réteg, residual-lal és anélkül (kb. fél perc).

In [ ]:
print(f"{'modell':>22} {'végső MSE':>10} {'|grad| az inp rétegen':>22}")
for depth in [10, 50]:
    for residual in [False, True]:
        keras.utils.set_random_seed(0)
        m = build_deep(depth, residual=residual)
        g = first_layer_grad_norm(m, X_xor, y_xor, mse)
        loss = fit_xor(m)[-1]
        name = f"{depth} réteg, {'residual' if residual else 'sima'}"
        RESULTS[f"1A {name}"] = (round(loss, 4), g)
        print(f"{name:>22} {loss:>10.4f} {g:>22.2e}")

**Mit látunk?** 50 rétegnél a sima háló gradiense a háló elején nagyságrendekkel kisebb, és a veszteség 0,25 körül ragad (ennyit ad a konstans 0,5 kimenet is). A residual ágon viszont a gradiens változatlanul visszafut, ezért a mély háló is tanul.

## 1B · Saját feladat: Highway-réteg (⏱ 10 perc)

A Highway-háló (19–20. dia) a residual ötletét **kapuval** finomítja: egy tanult kapu (`T`, 0 és 1 között) dönti el, mennyit vegyen az új jelből (`H`), és mennyit engedjen át változatlanul a régiből:

`h ← H · T + h · (1 − T)`, ahol `H = tanh(W_H·h + b_H)` és `T = sigmoid(W_T·h + b_T)`.

A kapu biasát **−3-ról** indítjuk: így kezdetben T ≈ 0,05, azaz a réteg szinte csak átengedi a bemenetet, és a mély háló is tanítható marad.

**Feladat.** Egészítsd ki a `build_highway` ciklusát az 1A `build_deep` mintájára:
1. `H`: `Dense(width, activation="tanh", name=f"H_{i}")`
2. `T`: `Dense(width, activation="sigmoid", bias_initializer=keras.initializers.Constant(gate_bias), name=f"T_{i}")`
3. `h = H * T + h * (1 - T)` (a Keras-tenzorokkal közvetlenül lehet számolni)

In [ ]:
def build_highway(depth, width=16, gate_bias=-3.0):
    inp = keras.Input(shape=(2,))
    h = layers.Dense(width, activation="tanh", name="inp")(inp)
    for i in range(depth):
        # TODO 1B: H, T és az új h
        raise NotImplementedError("TODO")
    out = layers.Dense(1, activation="sigmoid", name="out")(h)
    return keras.Model(inp, out)

In [ ]:
check_highway()

**Kísérlet:** 50 rétegnél hasonlítsd össze a sima, a residual és a Highway-hálót (kb. fél perc). Csak futtasd.

In [ ]:
print(f"{'modell':>22} {'végső MSE':>10} {'|grad| az inp rétegen':>22}")
for name, builder in [("50 réteg, sima", lambda: build_deep(50)),
                      ("50 réteg, residual", lambda: build_deep(50, residual=True)),
                      ("50 réteg, highway", lambda: build_highway(50))]:
    keras.utils.set_random_seed(0)
    m = builder()
    g = first_layer_grad_norm(m, X_xor, y_xor, mse)
    loss = fit_xor(m)[-1]
    RESULTS[f"1B {name}"] = (round(loss, 4), g)
    print(f"{name:>22} {loss:>10.4f} {g:>22.2e}")

**Kérdések:**
1. Melyik háló tanult meg a XOR-t 50 rétegnél?
2. Miért kell a kapu biasát negatívról indítani? (Tipp: mekkora T kezdetben, és mi megy át?)
3. Melyik mai modellben láttál hasonló kapuzást? (21–22. dia)

*Válaszaid:* …

**Bónusz:** futtasd a Highway-hálót `gate_bias=0.0`-val is! Mit tapasztalsz?

In [ ]:
# Bónusz: gate_bias = 0
keras.utils.set_random_seed(0)
m = build_highway(50, gate_bias=0.0)
print("gate_bias=0 → végső MSE:", round(fit_xor(m)[-1], 4))

## 🗣 Közös pont 1 · Mini verseny: a legmélyebb tanítható háló

**Szabály:** XOR, `width=16`, 200 epoch. Az nyer, akinek a **legmélyebb** hálója eléri az **MSE < 0,05**-öt. Választhatsz sima, residual vagy Highway hálót, és a Highwaynél a `gate_bias`-t is állíthatod. A cella fusson le kb. 1 perc alatt (100–200 rétegnél ez már határeset).

Állítsd be a paramétereket, futtasd, és a „Beküldendő” sort másold be a közös táblázatba!

In [ ]:
MY_NAME   = "Név"
DEPTH     = 100           # rejtett rétegek száma
KIND      = "highway"     # "sima", "residual" vagy "highway"
GATE_BIAS = -3.0          # csak highway esetén számít

keras.utils.set_random_seed(0)
if KIND == "highway":
    m = build_highway(DEPTH, gate_bias=GATE_BIAS)
else:
    m = build_deep(DEPTH, residual=(KIND == "residual"))
final = fit_xor(m)[-1]
print(f"Beküldendő: {MY_NAME}; {KIND}; mélység={DEPTH}; gate_bias={GATE_BIAS}; MSE={final:.4f}; "
      f"{'SIKERES' if final < 0.05 else 'nem sikerült'}")

---
# 2. feladat · Ne lásson a jövőbe: maszkolt self-attention

A kauzális konvolúció (29–31. dia) és a maszkolt self-attention (32. dia) ugyanazt biztosítja: a t-edik kimenet csak a ≤ t pozíciókból számolódhat.

## 2A · Kidolgozott példa: kumulatív átlag maszkolt softmaxszal

**1. lépés: a maszk.** Egy `n × n`-es igaz/hamis mátrix: `True` ott, ahol a j-edik oszlop a i-edik sorhoz képest **jövő** (j > i). Ezeket a helyeket kell eltakarni.

In [ ]:
def causal_mask(n):
    i = ops.arange(n)[:, None]      # sorindex, oszlopvektorként
    j = ops.arange(n)[None, :]      # oszlopindex, sorvektorként
    return j > i                    # True = jövő → maszkolni kell

print(ops.convert_to_numpy(causal_mask(5)).astype(int))

**2. lépés: maszkolt softmax.** Vegyünk csupa 0 pontszámot. A jövő helyére `−∞` kerül, a softmax után ott a súly 0, a többi pozíció pedig **egyenlő** súlyt kap.

In [ ]:
scores = ops.zeros((5, 5))
scores = ops.where(causal_mask(5), float("-inf"), scores)
w = ops.softmax(scores, axis=-1)
print(np.round(ops.convert_to_numpy(w), 2))

**3. lépés: súlyozott összeg.** A súlymátrixot a sorozattal szorozva minden pozícióban az eddigi elemek átlagát kapjuk: ez egy kauzális mozgó átlag.

In [ ]:
x = np.array([[2.], [4.], [6.], [8.], [10.]], "float32")      # (n=5, d=1)
y = ops.convert_to_numpy(ops.matmul(w, x)).flatten()
print("maszkolt softmax:", y)
print("kumulatív átlag :", np.cumsum(x.flatten()) / np.arange(1, 6))

**Összefoglalva a minta:** `pontszámok → maszk (−∞ a jövőben) → softmax → súlyok @ értékek`. A valódi attentionben csak a pontszám más: nem 0, hanem a lekérdezés és a kulcs hasonlósága, `QKᵀ / √d`.

## 2B · Saját feladat: maszkolt self-attention (⏱ 10 perc)

`Attn(Q, K, V) = softmax( QKᵀ/√d + maszk ) · V`

**Feladat.** Írd meg a `masked_attention(q, k, v)` függvényt a 2A mintájára:
1. pontszámok: `ops.matmul(q, ops.swapaxes(k, -1, -2)) / math.sqrt(d)`
2. maszk: `ops.where(causal_mask(n), float("-inf"), scores)`
3. súlyok: `ops.softmax(..., axis=-1)`
4. kimenet: `ops.matmul(w, v)`

A függvény adja vissza a kimenetet **és** a súlyokat is: `return out, w`.

In [ ]:
def masked_attention(q, k, v):
    # q, k, v alakja: (..., n, d)
    n, d = ops.shape(q)[-2], ops.shape(q)[-1]
    # TODO 2B: pontszámok, maszk, softmax, kimenet
    raise NotImplementedError("TODO")
    return out, w

In [ ]:
check_attention()

**Nézzük meg a súlyokat** a 32. dia mondatán (véletlen beágyazásokkal, tehát a számok mások, a minta ugyanaz).

In [ ]:
tokens = ["A", "kutya", "ugat", "a", "kertben"]
e = np.random.default_rng(3).normal(size=(1, 5, 16)).astype("float32")
_, w = masked_attention(e, e, e)
w = ops.convert_to_numpy(w)[0]

plt.figure(figsize=(4, 3.5))
plt.imshow(w, cmap="Blues", vmin=0, vmax=1)
plt.xticks(range(5), tokens, rotation=45); plt.yticks(range(5), tokens)
for i in range(5):
    for j in range(5):
        plt.text(j, i, f"{w[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.xlabel("kulcs"); plt.ylabel("lekérdezés"); plt.show()

**Kauzalitásteszt.** Ha a 12. pozíció utáni bemenetet véletlenre cseréljük, a 0–12. pozíció kimenete nem változhat.

In [ ]:
x = np.random.default_rng(4).normal(size=(1, 30, 16)).astype("float32")
assert check_causal(lambda z: masked_attention(z, z, z)[0], x, t0=12)
ok("a maszkolt attention nem lát a jövőbe")

**Bónusz: lokális (ablakos) maszk.** Írj egy `local_mask(n, w)` függvényt, amely a jövőt **és** a w-nél régebbi múltat is eltakarja (egy pozíció csak saját magát és az előző w − 1 pozíciót látja). Ez pontosan olyan, mint egy w szélességű kauzális konvolúció látótere. Tipp: `(j > i) | (j <= i - w)`.

In [ ]:
def local_mask(n, w):
    i = ops.arange(n)[:, None]
    j = ops.arange(n)[None, :]
    # TODO bónusz
    raise NotImplementedError("TODO")

check_local_mask()
print(ops.convert_to_numpy(local_mask(6, 3)).astype(int))

## 🗣 Közös pont 2 · Tapasztalatcsere: mennyibe kerül az attention?

Az attention pontszám-mátrixa n × n-es, ezért a futásidő n²-tel nő: ha n duplázódik, az idő kb. **négyszeres** lesz. Mérd meg a saját gépeden, és oszd meg az eredményt! A közös táblázatban látszani fog, mennyit számít a GPU, és hogy mindenkinél kb. 4 körül van-e az arány.

Csak futtasd (kb. 20 másodperc).

In [ ]:
import time
MY_NAME = "Név"

def time_attention(n, d=64, repeat=3):
    x = tf.random.normal((1, n, d))
    ops.convert_to_numpy(masked_attention(x, x, x)[0])          # bemelegítés
    best = float("inf")
    for _ in range(repeat):
        t0 = time.perf_counter()
        ops.convert_to_numpy(masked_attention(x, x, x)[0])      # a numpy-ra alakítás megvárja a GPU-t
        best = min(best, time.perf_counter() - t0)
    return best

t2k, t4k = time_attention(2048), time_attention(4096)
device = "GPU" if tf.config.list_physical_devices("GPU") else "CPU"
RESULTS["2 attention-idő 4096 / 2048 arány"] = round(t4k / t2k, 2)
print(f"Beküldendő: {MY_NAME}; {device}; n=2048: {t2k*1000:.1f} ms; n=4096: {t4k*1000:.1f} ms; arány: {t4k/t2k:.2f}")

---
# 3. feladat · Knowledge distillation

Egy nagy, jól tanított **teacher** háló tudását átadjuk egy kicsi **student** hálónak (46. dia: Parallel WaveNet, 47. dia: mai kis LLM-ek).

## 3A · Kidolgozott példa: tanulás a teacher puha címkéiből

**Az adat:** két összefonódó félhold (2D, két osztály). Csak **20 címkézett** pont van, és sok címke nélküli.

In [ ]:
from sklearn.datasets import make_moons, load_digits
from sklearn.model_selection import train_test_split

Xm, ym = make_moons(n_samples=800, noise=0.25, random_state=0)
Xm = Xm.astype("float32")
Xm_pool, Xm_te, ym_pool, ym_te = train_test_split(Xm, ym, test_size=200, random_state=0, stratify=ym)
lab = np.concatenate([np.nonzero(ym_pool == c)[0][:10] for c in range(2)])      # osztályonként 10
Xm_lab, ym_lab = Xm_pool[lab], ym_pool[lab]
print("címkézett:", len(Xm_lab), "| címke nélküli:", len(Xm_pool), "| teszt:", len(Xm_te))

**A teacher: honnan vannak a súlyai?**

A valóságban a teacher egy **már betanított, nagy modell**: valaki korábban sok adaton, sok számítással tanította, mi pedig készen kapjuk. A desztillációhoz a teacher belsejét (súlyait, architektúráját) nem kell ismernünk; elég, ha a **kimeneteit** le tudjuk kérdezni. A student soha nem kapja meg a teacher súlyait, csak a válaszait.

Ebben a példában nincs kész teacherünk, ezért **itt helyben tanítjuk be**, és így szimuláljuk az „előre betanított” modellt:

- **Adat:** az összes 600 pool-pont **a címkéivel együtt**. A teacher tehát sokkal több címkét látott, mint amennyi a studentnek jut (20). Ez a valóságot modellezi: a nagy modellt valaki sok címkézett adaton tanította.
- **Architektúra:** 2 → 64 → 64 → 2, ReLU rejtett rétegekkel, kb. 4500 paraméter (a student csak 2 → 8 → 2, 42 paraméter).
- **Tanítás:** Adam, 200 epoch, teljes batch, kereszt-entrópia.
- **Amit nem látott:** a 200 tesztpontot. Ezeken mérjük mindkét háló pontosságát.
- **Kimenet:** **logit** (nincs softmax a végén); a valószínűségeket mi számoljuk belőle a következő lépésben.

In [ ]:
def accuracy(model, X, y):
    return float(np.mean(np.argmax(ops.convert_to_numpy(model(X)), -1) == y))

keras.utils.set_random_seed(0)
teacher_m = keras.Sequential([keras.Input(shape=(2,)), layers.Dense(64, activation="relu"),
                              layers.Dense(64, activation="relu"), layers.Dense(2)])
teacher_m.compile(optimizer=keras.optimizers.Adam(1e-2),
                  loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True))
teacher_m.fit(Xm_pool, ym_pool, epochs=200, batch_size=len(Xm_pool), verbose=0)
print("teacher teszt-pontosság:", accuracy(teacher_m, Xm_te, ym_te))

**Mit tudunk a teacherről?** Nézzük meg a méretét és a döntési határát. A student ugyanezt a határt próbálja majd megtanulni, de csak a teacher válaszaiból, és sokkal kevesebb paraméterrel.

In [ ]:
print("teacher paraméterei:", teacher_m.count_params())
teacher_m.summary()

xx, yy = np.meshgrid(np.linspace(-1.5, 2.5, 200), np.linspace(-1, 1.5, 200))
grid = np.c_[xx.ravel(), yy.ravel()].astype("float32")
p1 = ops.convert_to_numpy(ops.softmax(teacher_m(grid), axis=-1))[:, 1].reshape(xx.shape)
plt.figure(figsize=(4.5, 3))
plt.contourf(xx, yy, p1, levels=20, cmap="coolwarm", alpha=0.6)
plt.scatter(Xm_te[:, 0], Xm_te[:, 1], c=ym_te, cmap="coolwarm", s=6, edgecolors="k", linewidths=0.2)
plt.colorbar(label="teacher: P(1. osztály)"); plt.title("A teacher döntési határa (tesztpontokkal)"); plt.show()

A színátmenet a teacher **bizonytalanságát** mutatja: a két félhold találkozásánál a valószínűség 0,5 körül van. Ez az információ a sima 0/1 címkékben nincs benne, a puha címkékben viszont igen.

**Puha címkék.** A teacher logitjaiból softmaxszal valószínűségeket kapunk. A határ közelében a teacher „bizonytalan” (pl. 0,6–0,4): ez az információ a sima 0/1 címkében nincs benne.

In [ ]:
p_teacher_m = ops.softmax(teacher_m.predict(Xm_pool, verbose=0), axis=-1)
print(np.round(ops.convert_to_numpy(p_teacher_m)[:6], 2))

**A veszteség: kereszt-entrópia puha címkékkel.** Ugyanaz, mint a szokásos kereszt-entrópia, csak a cél nem egy 0/1 vektor, hanem a teacher eloszlása:

`L = − átlag( Σ_c p_teacher[c] · log p_student[c] )`

In [ ]:
def soft_ce(student_logits, p_teacher):
    log_p_student = ops.log_softmax(student_logits, axis=-1)
    return -ops.mean(ops.sum(p_teacher * log_p_student, axis=-1))

**Saját tanítóciklus.** A `GradientTape` felveszi a számítást, és kiszámolja a gradienseket. A student vagy a 20 valódi címkén tanul, vagy a teacher puha címkéin az összes címke nélküli ponton.

In [ ]:
def train_student_moons(use_teacher, seed=0, steps=300):
    keras.utils.set_random_seed(seed)
    student = keras.Sequential([keras.Input(shape=(2,)), layers.Dense(8, activation="relu"), layers.Dense(2)])
    opt = keras.optimizers.Adam(1e-2)
    ce = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    for _ in range(steps):
        with tf.GradientTape() as tape:
            if use_teacher:
                loss = soft_ce(student(Xm_pool, training=True), p_teacher_m)
            else:
                loss = ce(ym_lab, student(Xm_lab, training=True))
        grads = tape.gradient(loss, student.trainable_variables)
        opt.apply_gradients(zip(grads, student.trainable_variables))
    return accuracy(student, Xm_te, ym_te)

In [ ]:
a_lab = np.mean([train_student_moons(False, s) for s in range(3)])
a_kd = np.mean([train_student_moons(True, s) for s in range(3)])
RESULTS["3A student, 20 címke"] = round(float(a_lab), 3)
RESULTS["3A student, teacher puha címkéi"] = round(float(a_kd), 3)
print(f"student, 20 valódi címke     : {a_lab:.3f}")
print(f"student, teacher puha címkéi : {a_kd:.3f}")

**Mit látunk?** A student, amely a teachertől tanult, jobb, mert a teacher a címke nélküli pontokra is ad „tanácsot”, és ez a tanács a bizonytalanságot is tartalmazza.

## 3B · Saját feladat: desztilláció hőmérséklettel (⏱ 10 perc)

Kézzel írt számjegyek (8×8 pixel, 10 osztály). A student csak **100 címkét** kap (osztályonként 10-et), és a teacher válaszait az összes tanító képre.

Két újdonság a 3A-hoz képest:
1. **Hőmérséklet (T):** a logitokat T-vel osztjuk a softmax előtt. T > 1 „ellágyítja” az eloszlást, így a kis valószínűségek (pl. „ez a 3-as kicsit 8-as”) is látszanak.
2. **Keverés:** a valódi címkék kereszt-entrópiáját is megtartjuk: `L = (1 − λ)·CE(címke) + λ·KD`.

A KD-tag (Hinton, 2015): `KD = T² · KL( softmax(z_teacher/T) ‖ softmax(z_student/T) )`. A T² szorzó a gradiensek nagyságrendjét állítja vissza.

**Az adat és a teacher** (csak futtasd). A teacher itt is helyben tanul, ugyanúgy, mint a 3A-ban:

- **Adat:** mind a kb. 1200 tanító kép, **címkékkel** (a student csak 100 címkét kap).
- **Architektúra:** 64 → 256 → 256 → 10, kb. 85 000 paraméter. A student 64 → 32 → 10, kb. 2400 paraméter, vagyis kb. 35-ször kisebb.
- **Tanítás:** Adam, 300 epoch, teljes batch. A tesztképeket nem látja.
- **Amit a student kap tőle:** csak a logitjait (`teacher_logits`) a tanító képekre, a súlyait nem.

In [ ]:
Xd, yd = load_digits(return_X_y=True)
Xd = (Xd / 16.0).astype("float32")
X_tr, X_te, y_tr, y_te = train_test_split(Xd, yd, test_size=0.33, random_state=0, stratify=yd)
lab = np.concatenate([np.nonzero(y_tr == c)[0][:10] for c in range(10)])
X_lab, y_lab = X_tr[lab], y_tr[lab]
print("tanító:", len(X_tr), "| címkézett:", len(X_lab), "| teszt:", len(X_te))

In [ ]:
keras.utils.set_random_seed(0)
teacher = keras.Sequential([keras.Input(shape=(64,)), layers.Dense(256, activation="relu"),
                            layers.Dense(256, activation="relu"), layers.Dense(10)])
teacher.compile(optimizer=keras.optimizers.Adam(1e-3),
                loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True))
teacher.fit(X_tr, y_tr, epochs=300, batch_size=len(X_tr), verbose=0)
teacher_logits = ops.convert_to_tensor(teacher.predict(X_tr, verbose=0))
RESULTS["3B teacher"] = round(accuracy(teacher, X_te, y_te), 3)
print("teacher teszt-pontosság:", RESULTS["3B teacher"])

**Feladat 3B.1.** Írd meg a `kd_loss` függvényt:
1. `p_teacher = ops.softmax(teacher_logits / T, axis=-1)`
2. `p_student = ops.softmax(student_logits / T, axis=-1)`
3. `keras.losses.KLDivergence()(p_teacher, p_student)` (ez a batchre átlagol), szorozva `T * T`-vel.

In [ ]:
def kd_loss(student_logits, teacher_logits, T):
    # TODO 3B.1
    raise NotImplementedError("TODO")

In [ ]:
check_kd_loss()

**Feladat 3B.2.** Egészítsd ki a tanítóciklust a 3A mintájára. Ha `use_kd` igaz, a veszteség:

`loss = (1 − lam) · ce(y_lab, student(X_lab)) + lam · kd_loss(student(X_tr), teacher_logits, T)`

Ha `use_kd` hamis, csak a címkés kereszt-entrópia marad.

In [ ]:
def train_student(use_kd, seed, T=4.0, lam=0.7, steps=300):
    keras.utils.set_random_seed(seed)
    student = keras.Sequential([keras.Input(shape=(64,)), layers.Dense(32, activation="relu"), layers.Dense(10)])
    opt = keras.optimizers.Adam(1e-2)
    ce = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    for _ in range(steps):
        with tf.GradientTape() as tape:
            # TODO 3B.2: a veszteség
            raise NotImplementedError("TODO")
        grads = tape.gradient(loss, student.trainable_variables)
        opt.apply_gradients(zip(grads, student.trainable_variables))
    return accuracy(student, X_te, y_te)

**Kísérlet** (csak futtasd, kb. fél perc): 5 seed átlaga, desztillációval és anélkül.

In [ ]:
base = [train_student(False, s) for s in range(5)]
kd = [train_student(True, s) for s in range(5)]
RESULTS["3B student, 100 címke"] = round(float(np.mean(base)), 3)
RESULTS["3B student, desztillációval"] = round(float(np.mean(kd)), 3)
print(f"student, 100 címke         : {np.mean(base):.3f} ± {np.std(base):.3f}")
print(f"student, desztillációval   : {np.mean(kd):.3f} ± {np.std(kd):.3f}")
print(f"teacher                    : {RESULTS['3B teacher']:.3f}")

## 🗣 Közös pont 3 · Mini verseny: a legjobb student

**Szabály:** a student mérete (32 neuron), a lépésszám (300) és a 100 címke kötött. Csak a **T** hőmérsékletet és a **λ** keverési arányt állíthatod. Az eredmény 3 seed átlaga a teszthalmazon, így a szerencse kevésbé számít.

Próbálj ki néhány beállítást, és a legjobbat küldd be! Tipp: mi történik T = 1-nél és T = 20-nál?

In [ ]:
MY_NAME = "Név"
T_MINE   = 4.0     # hőmérséklet
LAM_MINE = 0.7     # 0 = csak címke, 1 = csak teacher

acc = np.mean([train_student(True, s, T=T_MINE, lam=LAM_MINE) for s in range(3)])
print(f"Beküldendő: {MY_NAME}; T={T_MINE}; λ={LAM_MINE}; pontosság={acc:.4f}")

---
## Zárás · Összefoglaló

Futtasd le a cellát, és nézd meg a mért számaidat egyben.

In [ ]:
for k, v in RESULTS.items():
    print(f"{k:40s} {v}")

| Építőelem | Amit ma kipróbáltál |
|---|---|
| Residual és Highway | a skip kapcsolat (kapuval vagy anélkül) megmenti a gradienst |
| Kauzális maszk | ugyanaz a minta mozgó átlagra és attentionre: −∞ a jövőben, softmax előtt |
| Desztilláció | a teacher puha eloszlása több tudást ad át, mint a címke |